In [11]:
import json
from typing import List
import base64
import filetype

from langchain_core.documents import Document
from test_unstructured.partition.html.test_convert import elements
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

def documents_partition(document_path:str):
    elements=partition_pdf(
        filename=document_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],#Grab images that found in the PDF
        extract_image_block_to_payload=True #store grabbed images as base64 data you can actually use
    )

    print(f"extracted {len(elements)} elements")
    return elements

file_path="./docs/attention-is-all-you-need.pdf"
elements=documents_partition(file_path)

def create_chunks(elements):
    chunks=chunk_by_title(
        elements=elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )

    print(f"{len(chunks)} chunks created")

    return chunks

chunks=create_chunks(elements)

def separate_content_types(chunk):
    content_data={
        'text':chunk.text,
        'tables':[],
        'images':[],
        'types':['text']
    }

    if hasattr(chunk,'metadata') and hasattr(chunk.metadata,'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type=type(element).__name__

            if element_type == 'Table':
                content_data['types'].append('table')
                table_html=getattr(element.metadata,'text_as_html',element.text)
                content_data['tables'].append(table_html)

            elif element_type == 'Image':
                if hasattr(element,'metadata') and hasattr(element.metadata,'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)

    # passing content_data['types'] list into set prevents duplicate items
    content_data['types']=list(set(content_data['types']))
    return content_data

# Identify the type of image
def _detect_mime_type(image_base64: str) -> str:

    try:
        raw = base64.b64decode(image_base64[:64] + "==", validate=False)
        kind = filetype.guess(raw)
        return kind.mime if kind else "image/jpeg"
    except Exception:
        return "image/jpeg"


def create_ai_summary(text: str, tables: List[str], images: List[str]) -> str:
    TEXT_MODEL = "openai/gpt-oss-20b"
    VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

    try:
        model_name = VISION_MODEL if images else TEXT_MODEL
        model = ChatGroq(model=model_name, temperature=0)

        prompt = f"""You are creating a searchable description for document content retrieval
                     CONTENT TO ANALYZE:
                     TEXT CONTENT:
                     {text}
                """

        if tables:
            prompt += "Tables:\n"
            for i, table in enumerate(tables):
                prompt += f"Table{i+1}:\n {table}\n\n"

        if images:
            prompt += f"\n{len(images)} image(s) are attached below for visual analysis.\n"

        prompt += """
                YOUR TASK:
                Generate a comprehensive, searchable breakdown of the content above. Prioritize informational density and findability over brevity. Your output must strictly follow this structure:

                1. CORE TOPICS & CONCEPTS: List the main themes, domains, and abstract ideas discussed.
                2. KEY FACTS, NUMBERS & DATA: Extract precise statistics, dates, entities, and critical data points found in the text and tables.
                3. POTENTIAL USER QUERIES: List 5-10 specific questions this exact content is qualified to answer perfectly.
                4. VISUAL & GRAPHICAL ANALYSIS: Synthesize any patterns, trends, or visual insights from the tables and/or attached images.
                5. ALTERNATIVE SEARCH KEYWORDS: Provide synonyms, industry jargon, acronyms, and variations of terms users might type to find this document.

                SEARCHABLE DESCRIPTION:
                """

        message_content = [{"type": "text", "text": prompt}]

        for image_base64 in images:
            if not image_base64:
                continue
            mime_type = _detect_mime_type(image_base64)
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:{mime_type};base64,{image_base64}"}
            })

        message = HumanMessage(content=message_content)
        summary = model.invoke([message])

        return summary.content

    except Exception as e:
        print(f"AI summary failed: {e}")
        return ""



def summarise_chunks(chunks):

    langchain_documents=[]

    for chunk in chunks:
        content_data=separate_content_types(chunk)

        if content_data['tables'] or content_data['images']:
            try:
                enhanced_data=create_ai_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )

                print(f"Enhanced content :{enhanced_data[:200]}")

            except Exception as e:
                print(f"AI summary fail :{e}")
                enhanced_data=content_data['text']

        else:
            enhanced_data=content_data['text']

        doc=Document(
            page_content=enhanced_data,
            metadata={
                "original_content":json.dumps({
                    "raw_text":content_data['text'],
                    "tables_html":content_data['tables'],
                    "images_base64":content_data['images']
                })
            }
        )

        langchain_documents.append(doc)

    print(f"Processed {len(langchain_documents)} chunks")

    return langchain_documents

docs=summarise_chunks(chunks)





extracted 215 elements
25 chunks created
Processed 25 chunks
